<a href="https://colab.research.google.com/github/DKavya8/chestxray-bias-audit/blob/main/notebooks/4_age_matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Negative-control resampler

This notebook addresses:

1. Building and running the identical exact-bin matching pipeline using randomly permuted fake age.
2. Testing whether fake-age matching changes the female-minus-male FNR gap.

## Conclusion

Fake-age matching did not materially change the sex gap.

Mean change in gap: 0.00020  
95% CI: −0.00153 to 0.00131  

The confidence interval includes zero, and all 10 split-specific confidence
intervals include zero. Therefore, there is no evidence that the matching
procedure itself introduced an artifact.

In [3]:
!pip install -q pandas pyarrow
from google.colab import drive
drive.mount('/content/drive')

import glob, os, sys

if not os.path.exists('/content/cxr'):
    !git clone -q https://github.com/DKavya8/chestxray-bias-audit.git /content/cxr
else:
    !cd /content/cxr && git pull -q
sys.path.insert(0, '/content/cxr')

# Find the required files whether the project is in My Drive,
# a Drive shortcut, or a Shared Drive.
metadata_matches = glob.glob('/content/drive/**/metadata_clean.csv', recursive=True)
scores_matches = glob.glob('/content/drive/**/densenet121_all_scores.parquet', recursive=True)

print('Metadata matches:')
print('\n'.join(metadata_matches) if metadata_matches else 'NOT FOUND')
print('\nScore-file matches:')
print('\n'.join(scores_matches) if scores_matches else 'NOT FOUND')

assert metadata_matches, (
    'metadata_clean.csv was not found. If the folder is under Shared with me, '
    'add a shortcut to it in My Drive, remount Drive, and rerun this cell.'
)
assert scores_matches, (
    'densenet121_all_scores.parquet was not found. Confirm that the shared '
    'project folder contains this file and is accessible from Colab.'
)

# Prefer files that live together in the same results directory.
pair = next(
    (
        (m, s) for m in metadata_matches for s in scores_matches
        if os.path.dirname(m) == os.path.dirname(s)
    ),
    None,
)
assert pair is not None, (
    'The metadata and score files were found, but not in the same directory. '
    'Move them into the same results folder or set their paths manually.'
)

META_PATH, SCORES_PATH = pair
RESULTS = os.path.dirname(META_PATH)
BASE = os.path.dirname(RESULTS)
SPLITS_ROOT = '/content/cxr/splits'

print('\nUsing:')
print('BASE        =', BASE)
print('RESULTS     =', RESULTS)
print('META_PATH   =', META_PATH)
print('SCORES_PATH =', SCORES_PATH)
print('SPLITS_ROOT =', SPLITS_ROOT)

for p in [META_PATH, SCORES_PATH, SPLITS_ROOT]:
    print(os.path.exists(p), p)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Metadata matches:
/content/drive/MyDrive/metadata_clean.csv

Score-file matches:
/content/drive/MyDrive/densenet121_all_scores.parquet

Using:
BASE        = /content/drive
RESULTS     = /content/drive/MyDrive
META_PATH   = /content/drive/MyDrive/metadata_clean.csv
SCORES_PATH = /content/drive/MyDrive/densenet121_all_scores.parquet
SPLITS_ROOT = /content/cxr/splits
True /content/drive/MyDrive/metadata_clean.csv
True /content/drive/MyDrive/densenet121_all_scores.parquet
True /content/cxr/splits


In [4]:
import pandas as pd, re
meta = pd.read_csv(META_PATH, dtype=str)
print("shape:", meta.shape)
print("columns:", list(meta.columns))
print("\n--- head ---"); print(meta.head(3).to_string())

def pick(cands): return next((c for c in cands if c in meta.columns), None)
COL_IMAGE   = pick(['Image Index','image_index'])
COL_PATIENT = pick(['Patient ID','patient_id'])
COL_SEX     = pick(['sex','Patient Sex','Patient Gender','Gender'])
COL_AGE     = pick(['age','Patient Age','patient_age','Age'])
print("\ndetected -> image:",COL_IMAGE,"| patient:",COL_PATIENT,"| sex:",COL_SEX,"| age:",COL_AGE)
assert all([COL_IMAGE,COL_PATIENT,COL_SEX,COL_AGE]), \
    "A needed column wasn't detected — paste the 'columns:' list and I'll map it."

pid = meta[COL_PATIENT].astype(str)
fu  = meta[COL_IMAGE].astype(str).str.extract(r'_(\d+)\.png$')[0].astype(float)
age = pd.to_numeric(meta[COL_AGE], errors='coerce')
t = pd.DataFrame({"pid":pid,"fu":fu,"age":age})
n_pat = t["pid"].nunique(); multi = (t.groupby("pid")["fu"].nunique()>1).sum()
spread = t.groupby("pid")["age"].agg(lambda s: s.max()-s.min())
print(f"\npatients: {n_pat}")
print(f"with >1 scan: {multi} ({100*multi/n_pat:.1f}%)")
print(f"age changes across scans: {(spread>0).sum()} (max spread {int(spread.max())} yrs)")

shape: (112106, 24)
columns: ['image_index', 'patient_id', 'age', 'sex', 'finding_labels', 'view_position', 'follow_up', 'pixel_spacing_x', 'pixel_spacing_y', 'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia', 'age_bin']

--- head ---
        image_index patient_id age sex          finding_labels view_position follow_up pixel_spacing_x pixel_spacing_y Atelectasis Cardiomegaly Effusion Infiltration Mass Nodule Pneumonia Pneumothorax Consolidation Edema Emphysema Fibrosis Pleural_Thickening Hernia age_bin
0  00000001_000.png          1  57   M            Cardiomegaly            PA         0           0.143           0.143           0            1        0            0    0      0         0            0             0     0         0        0                  0      0   55-59
1  00000001_001.png          1  58   M  Cardiomegaly|Emphysema            PA   

In [5]:
import numpy as np
sex = meta[COL_SEX].astype(str).str.strip().str.upper().str[0]
work = pd.DataFrame({
    "Image Index": meta[COL_IMAGE].astype(str).str.strip(),
    "Patient ID":  meta[COL_PATIENT].astype(str).str.strip(),
    "sex": sex,
    "age": pd.to_numeric(meta[COL_AGE], errors="coerce"),
})
work["followup"] = work["Image Index"].str.extract(r'_(\d+)\.png$')[0].astype(float)

before = len(work)
work = work[work["sex"].isin(["F","M"]) & work["age"].notna()].copy()
print(f"dropped {before-len(work)} rows missing age/sex; {len(work)} remain")

base = (work.sort_values(["Patient ID","followup"])
             .groupby("Patient ID", as_index=False).first())
base["age"]   = base["age"].astype(int)
base["bin5"]  = 5  * (base["age"] // 5)
base["bin10"] = 10 * (base["age"] // 10)

assert base["Patient ID"].is_unique, "more than one row per patient!"
assert base["age"].notna().all() and set(base["sex"]) <= {"F","M"}
print("patients in base table:", len(base))
print("sex counts:\n", base["sex"].value_counts())
print("age -> min",base.age.min(),"max",base.age.max(),"median",int(base.age.median()))
print("\n5-yr bin x sex counts (watch the sparse bins):")
print(base.pivot_table(index="bin5", columns="sex", values="age", aggfunc="count", fill_value=0))

dropped 0 rows missing age/sex; 112106 remain
patients in base table: 30797
sex counts:
 sex
M    16625
F    14172
Name: count, dtype: int64
age -> min 1 max 95 median 47

5-yr bin x sex counts (watch the sparse bins):
sex      F     M
bin5            
0       98   116
5      162   211
10     237   308
15     388   484
20     787   891
25     948  1049
30    1155  1262
35    1223  1240
40    1453  1443
45    1671  1668
50    1755  1923
55    1593  1926
60    1200  1669
65     752  1159
70     396   724
75     232   373
80      87   132
85      28    40
90       6     7
95       1     0


In [6]:
# ---------------------------------------------------------
# Negative-control fake age (frozen patient-level permutation)
# ---------------------------------------------------------
FAKE_AGE_SEED = 20260825
fake_age_rng = np.random.default_rng(FAKE_AGE_SEED)

# Preserve the marginal age distribution while breaking the link between
# a patient's age and that patient's sex, labels, and model predictions.
base["fake_age"] = fake_age_rng.permutation(base["age"].to_numpy())
base["fake_bin5"] = 5 * (base["fake_age"] // 5)
base["fake_bin10"] = 10 * (base["fake_age"] // 10)

# Required invariants and diagnostics.
assert np.array_equal(
    np.sort(base["age"].to_numpy()),
    np.sort(base["fake_age"].to_numpy()),
), "Fake age must be an exact permutation of real age"
assert base["Patient ID"].is_unique

same_age_fraction = (base["age"] == base["fake_age"]).mean()
age_correlation = base["age"].corr(base["fake_age"])
print(f"fake-age seed: {FAKE_AGE_SEED}")
print(f"same-age fraction: {same_age_fraction:.4f}")
print(f"real/fake correlation: {age_correlation:.4f}")
print("age distribution preserved:", base["age"].describe().equals(base["fake_age"].describe()))
print("\nfake age by sex:")
print(base.groupby("sex")["fake_age"].describe().round(2))

fake-age seed: 20260825
same-age fraction: 0.0166
real/fake correlation: -0.0031
age distribution preserved: True

fake age by sex:
       count   mean    std  min   25%   50%   75%   max
sex                                                    
F    14172.0  45.86  16.73  1.0  34.0  47.0  58.0  94.0
M    16625.0  45.88  16.66  1.0  34.0  47.0  58.0  95.0


In [7]:
import numpy as np, pandas as pd

MATCH_SEED_BASE = 20260823
N_MATCH_SEEDS   = 100
MATCH_SEEDS = np.random.default_rng(MATCH_SEED_BASE).integers(0, 2**32, size=N_MATCH_SEEDS).tolist()
print("first 5 matching seeds:", MATCH_SEEDS[:5], "...")

def exact_bin_match(df, bin_col, seed):
    """One matched replicate. Per bin keep min(N_f,N_m): smaller sex kept whole,
    larger sex randomly sampled down to that count. Bins with a missing sex -> all dropped."""
    rng = np.random.default_rng(seed)
    keep = []
    for b, g in df.groupby(bin_col):
        f = g[g["sex"] == "F"]; m = g[g["sex"] == "M"]
        n = min(len(f), len(m))
        if n == 0:
            continue
        keep.append(f if len(f) == n else f.sample(n=n, random_state=rng.integers(0, 2**32)))
        keep.append(m if len(m) == n else m.sample(n=n, random_state=rng.integers(0, 2**32)))
    return pd.concat(keep, ignore_index=True) if keep else df.iloc[0:0].copy()


seed_dir = f"{SPLITS_ROOT}/seed_{3658676649}"
test_ids = set(pd.read_csv(f"{seed_dir}/test_patients.csv", dtype=str)["Patient ID"].str.strip())
split_df = base[base["Patient ID"].isin(test_ids)].copy()
print(f"\nsplit test patients: {len(split_df)}  (F={sum(split_df.sex=='F')}, M={sum(split_df.sex=='M')})")

matched = exact_bin_match(split_df, "bin5", MATCH_SEEDS[0])
print(f"matched patients:   {len(matched)}  (F={sum(matched.sex=='F')}, M={sum(matched.sex=='M')})")
print("per-bin F==M after matching?",
      (matched.groupby('bin5')['sex'].apply(lambda s:(s=='F').sum()==(s=='M').sum())).all())
print("dropped (unmatched):", len(split_df) - len(matched))

first 5 matching seeds: [3716687774, 177135906, 76975675, 752310521, 2253019170] ...

split test patients: 6158  (F=2789, M=3369)
matched patients:   5560  (F=2780, M=2780)
per-bin F==M after matching? True
dropped (unmatched): 598


In [8]:
import time
ALL_SEEDS = [int(s) for s in open(f"{SPLITS_ROOT}/seeds.txt").read().split()]
print("split seeds:", ALL_SEEDS)

def load_split_test(split_seed):
    p = f"{SPLITS_ROOT}/seed_{split_seed}/test_patients.csv"
    ids = set(pd.read_csv(p, dtype=str)["Patient ID"].str.strip())
    return base[base["Patient ID"].isin(ids)].copy()

matched_sets = {}
t0 = time.time()
for ss in ALL_SEEDS:
    split_df = load_split_test(ss)
    reps = [exact_bin_match(split_df, "bin10", ms) for ms in MATCH_SEEDS]
    matched_sets[ss] = reps
    sizes = [len(r) for r in reps]
    print(f"seed {ss}: split={len(split_df):5d}  matched≈{int(np.mean(sizes))}  "
          f"(min {min(sizes)}, max {max(sizes)})  F==M all reps: "
          f"{all((r.sex=='F').sum()==(r.sex=='M').sum() for r in reps)}")
print(f"\ndone in {time.time()-t0:.1f}s  |  {len(ALL_SEEDS)} splits × {N_MATCH_SEEDS} seeds = "
      f"{len(ALL_SEEDS)*N_MATCH_SEEDS} matched replicates")

split seeds: [3658676649, 768519171, 113462462, 2748406118, 1569714665, 2006902500, 342858866, 1591287646, 2763601433, 1524358342]
seed 3658676649: split= 6158  matched≈5576  (min 5576, max 5576)  F==M all reps: True
seed 768519171: split= 6160  matched≈5500  (min 5500, max 5500)  F==M all reps: True
seed 113462462: split= 6159  matched≈5500  (min 5500, max 5500)  F==M all reps: True
seed 2748406118: split= 6160  matched≈5746  (min 5746, max 5746)  F==M all reps: True
seed 1569714665: split= 6160  matched≈5694  (min 5694, max 5694)  F==M all reps: True
seed 2006902500: split= 6159  matched≈5642  (min 5642, max 5642)  F==M all reps: True
seed 342858866: split= 6159  matched≈5656  (min 5656, max 5656)  F==M all reps: True
seed 1591287646: split= 6158  matched≈5720  (min 5720, max 5720)  F==M all reps: True
seed 2763601433: split= 6160  matched≈5732  (min 5732, max 5732)  F==M all reps: True
seed 1524358342: split= 6161  matched≈5622  (min 5622, max 5622)  F==M all reps: True

done in 16.

In [9]:
# ---------------------------------------------------------
# Negative-control matching: same splits and same matching seeds
# ---------------------------------------------------------
fake_matched_sets_10yr = {}
fake_matched_sets_5yr = {}
t0 = time.time()

for ss in ALL_SEEDS:
    split_df = load_split_test(ss)
    reps10 = [exact_bin_match(split_df, "fake_bin10", ms) for ms in MATCH_SEEDS]
    reps5 = [exact_bin_match(split_df, "fake_bin5", ms) for ms in MATCH_SEEDS]
    fake_matched_sets_10yr[ss] = reps10
    fake_matched_sets_5yr[ss] = reps5

    ok10 = all((r.sex == "F").sum() == (r.sex == "M").sum() for r in reps10)
    ok5 = all((r.sex == "F").sum() == (r.sex == "M").sum() for r in reps5)
    n10 = [len(r) for r in reps10]
    n5 = [len(r) for r in reps5]
    print(
        f"seed {ss}: split={len(split_df):5d} | "
        f"fake10={int(np.mean(n10)):5d} (F==M: {ok10}) | "
        f"fake5={int(np.mean(n5)):5d} (F==M: {ok5})"
    )

assert len(fake_matched_sets_10yr) == len(ALL_SEEDS)
assert len(fake_matched_sets_5yr) == len(ALL_SEEDS)
assert all(len(v) == N_MATCH_SEEDS for v in fake_matched_sets_10yr.values())
assert all(len(v) == N_MATCH_SEEDS for v in fake_matched_sets_5yr.values())
print(
    f"\ndone in {time.time()-t0:.1f}s | "
    f"{len(ALL_SEEDS) * N_MATCH_SEEDS} fake-10yr + "
    f"{len(ALL_SEEDS) * N_MATCH_SEEDS} fake-5yr replicates"
)

seed 3658676649: split= 6158 | fake10= 5544 (F==M: True) | fake5= 5536 (F==M: True)
seed 768519171: split= 6160 | fake10= 5578 (F==M: True) | fake5= 5574 (F==M: True)
seed 113462462: split= 6159 | fake10= 5536 (F==M: True) | fake5= 5530 (F==M: True)
seed 2748406118: split= 6160 | fake10= 5782 (F==M: True) | fake5= 5720 (F==M: True)
seed 1569714665: split= 6160 | fake10= 5752 (F==M: True) | fake5= 5720 (F==M: True)
seed 2006902500: split= 6159 | fake10= 5648 (F==M: True) | fake5= 5644 (F==M: True)
seed 342858866: split= 6159 | fake10= 5716 (F==M: True) | fake5= 5710 (F==M: True)
seed 1591287646: split= 6158 | fake10= 5712 (F==M: True) | fake5= 5688 (F==M: True)
seed 2763601433: split= 6160 | fake10= 5726 (F==M: True) | fake5= 5696 (F==M: True)
seed 1524358342: split= 6161 | fake10= 5726 (F==M: True) | fake5= 5716 (F==M: True)

done in 32.3s | 1000 fake-10yr + 1000 fake-5yr replicates


In [10]:
def bin_is_empty_any_sex(df, bin_col):
    """Protocol trigger: True if ANY sex×bin cell in this split is empty."""
    tab = df.pivot_table(index=bin_col, columns="sex", values="age",
                         aggfunc="count", fill_value=0)
    for s in ("F","M"):
        if s not in tab.columns: return True
    return (tab[["F","M"]] == 0).any().any()

def ipw_weights(df):
    """Frozen IPW. Returns (weighted_df, chosen_bin_col, diagnostics).
    Target Q_b = (F-prop_b + M-prop_b)/2 ; weight = Q_b / observed-prop within own sex.
    If any 5-yr sex×bin is empty -> switch whole analysis to 10-yr bins."""
    bin_col = "bin5"
    if bin_is_empty_any_sex(df, "bin5"):
        bin_col = "bin10"
        print("  [frozen rule] empty 5-yr sex×bin found -> using 10-yr bins for IPW")

    d = df.copy()
    f_prop = d[d.sex=="F"][bin_col].value_counts(normalize=True)
    m_prop = d[d.sex=="M"][bin_col].value_counts(normalize=True)
    all_bins = sorted(set(f_prop.index) | set(m_prop.index))
    f_prop = f_prop.reindex(all_bins, fill_value=0.0)
    m_prop = m_prop.reindex(all_bins, fill_value=0.0)

    target = (f_prop + m_prop) / 2.0
    assert abs(target.sum() - 1.0) < 1e-9

    def w(row):
        obs = (f_prop if row.sex=="F" else m_prop)[row[bin_col]]
        return target[row[bin_col]] / obs if obs > 0 else np.nan
    d["weight"] = d.apply(w, axis=1)


    diag = {"bin_col": bin_col, "target_sums_to_1": float(target.sum())}
    for s in ("F","M"):
        wser = d[d.sex==s]["weight"]
        kish = (wser.sum()**2) / (wser**2).sum()
        diag[f"max_weight_{s}"] = float(wser.max())
        diag[f"kish_ESS_{s}"]   = float(kish)
        diag[f"n_{s}"]          = int(len(wser))
    return d, bin_col, diag

split_df = load_split_test(3658676649)
wdf, used_bin, diag = ipw_weights(split_df)
print("weighted patients:", len(wdf), "(everyone kept — no dropping)")

import json; print(json.dumps(diag, indent=2))
chk = (wdf.assign(wsum=wdf.weight)
          .pivot_table(index=used_bin, columns="sex", values="wsum", aggfunc="sum"))
chk = chk / chk.sum()
print("\nweighted F vs M bin distributions (should be ~identical):")
print(chk.round(4))

  [frozen rule] empty 5-yr sex×bin found -> using 10-yr bins for IPW
weighted patients: 6158 (everyone kept — no dropping)
{
  "bin_col": "bin10",
  "target_sums_to_1": 1.0000000000000002,
  "max_weight_F": 1.264926090828139,
  "kish_ESS_F": 2771.7308899743684,
  "n_F": 2789,
  "max_weight_M": 1.0724983422186722,
  "kish_ESS_M": 3352.450551510689,
  "n_M": 3369
}

weighted F vs M bin distributions (should be ~identical):
sex         F       M
bin10                
0      0.0157  0.0158
10     0.0459  0.0459
20     0.1252  0.1252
30     0.1649  0.1649
40     0.2020  0.2020
50     0.2288  0.2288
60     0.1522  0.1522
70     0.0567  0.0567
80     0.0085  0.0085
90     0.0002     NaN


In [11]:
ipw_sets = {}
ipw_diags = {}
for ss in ALL_SEEDS:
    split_df = load_split_test(ss)
    print(f"seed {ss}:", end=" ")
    wdf, used_bin, diag = ipw_weights(split_df)
    ipw_sets[ss] = wdf
    ipw_diags[ss] = diag
    print(f"n={len(wdf)}  bins={used_bin}  "
          f"maxW: F={diag['max_weight_F']:.2f}/M={diag['max_weight_M']:.2f}  "
          f"ESS: F={diag['kish_ESS_F']:.0f}/M={diag['kish_ESS_M']:.0f}")

print("\n--- diagnostics summary across splits ---")
maxw = max(max(d['max_weight_F'], d['max_weight_M']) for d in ipw_diags.values())
print(f"largest weight anywhere: {maxw:.2f}")
print(f"all splits used: {set(d['bin_col'] for d in ipw_diags.values())}")

seed 3658676649:   [frozen rule] empty 5-yr sex×bin found -> using 10-yr bins for IPW
n=6158  bins=bin10  maxW: F=1.26/M=1.07  ESS: F=2772/M=3352
seed 768519171:   [frozen rule] empty 5-yr sex×bin found -> using 10-yr bins for IPW
n=6160  bins=bin10  maxW: F=1.42/M=1.14  ESS: F=2759/M=3343
seed 113462462: n=6159  bins=bin5  maxW: F=1.37/M=1.17  ESS: F=2741/M=3359
seed 2748406118: n=6160  bins=bin5  maxW: F=2.05/M=1.63  ESS: F=2873/M=3250
seed 1569714665: n=6160  bins=bin5  maxW: F=1.38/M=1.78  ESS: F=2847/M=3256
seed 2006902500: n=6159  bins=bin5  maxW: F=1.77/M=1.15  ESS: F=2794/M=3308
seed 342858866: n=6159  bins=bin5  maxW: F=3.32/M=1.65  ESS: F=2818/M=3271
seed 1591287646: n=6158  bins=bin5  maxW: F=1.31/M=1.10  ESS: F=2836/M=3274
seed 2763601433: n=6160  bins=bin5  maxW: F=1.81/M=1.65  ESS: F=2835/M=3263
seed 1524358342: n=6161  bins=bin5  maxW: F=1.46/M=1.17  ESS: F=2822/M=3261

--- diagnostics summary across splits ---
largest weight anywhere: 3.32
all splits used: {'bin10', 'bi

In [12]:
def any_empty_5yr(seeds):
    for ss in seeds:
        if bin_is_empty_any_sex(load_split_test(ss), "bin5"):
            return True
    return False

GLOBAL_BIN = "bin10" if any_empty_5yr(ALL_SEEDS) else "bin5"
print(f"[global lock] IPW bin width for ALL splits: {GLOBAL_BIN}\n")

def ipw_weights_fixed(df, bin_col):
    d = df.copy()
    f_prop = d[d.sex=="F"][bin_col].value_counts(normalize=True)
    m_prop = d[d.sex=="M"][bin_col].value_counts(normalize=True)
    all_bins = sorted(set(f_prop.index) | set(m_prop.index))
    f_prop = f_prop.reindex(all_bins, fill_value=0.0)
    m_prop = m_prop.reindex(all_bins, fill_value=0.0)
    target = (f_prop + m_prop) / 2.0
    d["weight"] = d.apply(lambda r: target[r[bin_col]] /
                          ((f_prop if r.sex=="F" else m_prop)[r[bin_col]]), axis=1)
    diag = {"bin_col": bin_col}
    for s in ("F","M"):
        w = d[d.sex==s]["weight"]
        diag[f"max_weight_{s}"] = float(w.max())
        diag[f"kish_ESS_{s}"] = float(w.sum()**2 / (w**2).sum())
    return d, diag

ipw_sets, ipw_diags = {}, {}
for ss in ALL_SEEDS:
    wdf, diag = ipw_weights_fixed(load_split_test(ss), GLOBAL_BIN)
    ipw_sets[ss] = wdf
    ipw_diags[ss] = diag
    print(f"seed {ss}: n={len(wdf)}  maxW F={diag['max_weight_F']:.2f}/M={diag['max_weight_M']:.2f}  "
          f"ESS F={diag['kish_ESS_F']:.0f}/M={diag['kish_ESS_M']:.0f}")
print(f"\nall splits on: {set(d['bin_col'] for d in ipw_diags.values())}  (should be one width)")

[global lock] IPW bin width for ALL splits: bin10

seed 3658676649: n=6158  maxW F=1.26/M=1.07  ESS F=2772/M=3352
seed 768519171: n=6160  maxW F=1.42/M=1.14  ESS F=2759/M=3343
seed 113462462: n=6159  maxW F=1.32/M=1.13  ESS F=2745/M=3363
seed 2748406118: n=6160  maxW F=1.47/M=1.63  ESS F=2880/M=3259
seed 1569714665: n=6160  maxW F=1.18/M=1.10  ESS F=2855/M=3262
seed 2006902500: n=6159  maxW F=1.77/M=1.11  ESS F=2799/M=3312
seed 342858866: n=6159  maxW F=1.33/M=1.65  ESS F=2834/M=3277
seed 1591287646: n=6158  maxW F=1.24/M=1.08  ESS F=2839/M=3278
seed 2763601433: n=6160  maxW F=1.26/M=1.65  ESS F=2843/M=3270
seed 1524358342: n=6161  maxW F=1.42/M=1.12  ESS F=2826/M=3265

all splits on: {'bin10'}  (should be one width)


In [13]:
import json, time
import numpy as np, pandas as pd
from inference import NIH_FINDING_NAMES

RESULTS_A = "/content/cxr/results/group_a_densenet"

scores = pd.read_parquet(SCORES_PATH)
scores["Image Index"] = scores["Image Index"].astype(str).str.strip()
miss = [f for f in NIH_FINDING_NAMES if f not in scores.columns]
assert not miss, f"scores parquet missing findings: {miss}"
S = scores[["Image Index", *NIH_FINDING_NAMES]].rename(columns={f: f"s_{f}" for f in NIH_FINDING_NAMES})

lab_col = next((c for c in ["Finding Labels","finding_labels","labels"] if c in meta.columns), None)
assert lab_col, f"no Finding Labels column; meta cols = {list(meta.columns)}"
L = pd.DataFrame({"Image Index": meta[COL_IMAGE].astype(str).str.strip()})
fl = meta[lab_col].fillna("").astype(str).str.split("|")
for f in NIH_FINDING_NAMES:
    L[f"y_{f}"] = fl.apply(lambda labs: int(f in labs))

G = (base.merge(S, on="Image Index", how="left", validate="one_to_one")
         .merge(L, on="Image Index", how="left", validate="one_to_one"))
THRESHOLDS = json.loads(open(f"{RESULTS_A}/thresholds_by_seed.json").read())

print("G rows (patients):", len(G))
print("missing scores  :", int(G[[f's_{f}' for f in NIH_FINDING_NAMES]].isna().any(axis=1).sum()))
print("missing labels  :", int(G[[f'y_{f}' for f in NIH_FINDING_NAMES]].isna().any(axis=1).sum()))
print("findings match  :", list(THRESHOLDS[list(THRESHOLDS)[0]]) == list(NIH_FINDING_NAMES))

G rows (patients): 30797
missing scores  : 0
missing labels  : 0
findings match  : False


In [14]:
Gpid = G.set_index("Patient ID")

def test_ids(seed):
    return set(pd.read_csv(f"{SPLITS_ROOT}/seed_{seed}/test_patients.csv", dtype=str)["Patient ID"].str.strip())
def test_frame(seed):
    return G[G["Patient ID"].isin(test_ids(seed))].copy()

def pooled_fnr(df, thr, w=None):
    """Micro-FNR pooled across the 14 findings, each at its own frozen threshold."""
    w = np.ones(len(df)) if w is None else np.asarray(w, float)
    fn = pos = 0.0
    for f in NIH_FINDING_NAMES:
        y = df[f"y_{f}"].to_numpy(); s = df[f"s_{f}"].to_numpy(); p = (y == 1)
        pos += (w * p).sum(); fn += (w * (p & (s < thr[f]))).sum()
    return fn / pos if pos > 0 else np.nan

def fnrs(df, thr, wcol=None):
    isF = df["sex"].to_numpy() == "F"
    w = df[wcol].to_numpy() if wcol else np.ones(len(df))
    return pooled_fnr(df[isF], thr, w[isF]), pooled_fnr(df[~isF], thr, w[~isF])   # (F, M)

ga = pd.read_csv(f"{RESULTS_A}/group_a_densenet_condition_1_original_strict.csv")
ga_gap = ga[(ga.finding=="NIH_14_pooled") & (ga.subgroup=="sex_gap:female-minus-male")].set_index("split_seed")["value"]

raw = {}
print(f"{'seed':>12} {'S_raw(B)':>10} {'GroupA_S':>10} {'diff':>10}")
for ss in ALL_SEEDS:
    ff, mm = fnrs(test_frame(ss), THRESHOLDS[str(ss)]); s = ff - mm; raw[ss] = (ff, mm, s)
    a = float(ga_gap.get(ss, np.nan))
    print(f"{ss:>12} {s:>10.6f} {a:>10.6f} {s-a:>+10.2e}")

        seed   S_raw(B)   GroupA_S       diff
  3658676649   0.018350   0.039330  -2.10e-02
   768519171   0.005474   0.019636  -1.42e-02
   113462462   0.061249   0.017221  +4.40e-02
  2748406118   0.060195   0.038646  +2.15e-02
  1569714665   0.005929   0.025733  -1.98e-02
  2006902500   0.007893  -0.001113  +9.01e-03
   342858866   0.016807   0.025493  -8.69e-03
  1591287646  -0.005659   0.033800  -3.95e-02
  2763601433   0.029540  -0.009160  +3.87e-02
  1524358342   0.071156   0.037716  +3.34e-02


In [15]:
def matched_fnrs(ss):
    thr = THRESHOLDS[str(ss)]; F=[]; M=[]
    for rep in matched_sets[ss]:
        sub = Gpid.loc[rep["Patient ID"].values]
        ff, mm = fnrs(sub, thr); F.append(ff); M.append(mm)
    return float(np.mean(F)), float(np.mean(M))

def ipw_fnrs(ss):
    thr = THRESHOLDS[str(ss)]
    sub = test_frame(ss).merge(ipw_sets[ss][["Patient ID","weight"]], on="Patient ID", how="inner")
    return fnrs(sub, thr, wcol="weight")

rows = []
print(f"{'seed':>12} {'S_raw':>8} {'S_match':>8} {'S_ipw':>8} {'dS_mat':>8} {'dS_ipw':>8}")
for ss in ALL_SEEDS:
    fr, mr, sr = raw[ss]
    fm, mm = matched_fnrs(ss); sm = fm - mm
    fi, mi = ipw_fnrs(ss);     si = fi - mi
    rows.append(dict(seed=ss, FNRf_raw=fr, FNRm_raw=mr, S_raw=sr,
                     FNRf_match=fm, FNRm_match=mm, S_match=sm,
                     FNRf_ipw=fi, FNRm_ipw=mi, S_ipw=si,
                     dS_match=sr-sm, dS_ipw=sr-si))
    print(f"{ss:>12} {sr:>8.4f} {sm:>8.4f} {si:>8.4f} {sr-sm:>+8.4f} {sr-si:>+8.4f}")
pts = pd.DataFrame(rows)

        seed    S_raw  S_match    S_ipw   dS_mat   dS_ipw
  3658676649   0.0183   0.0086   0.0097  +0.0097  +0.0086
   768519171   0.0055  -0.0049  -0.0044  +0.0104  +0.0099
   113462462   0.0612   0.0519   0.0499  +0.0094  +0.0113
  2748406118   0.0602   0.0532   0.0532  +0.0069  +0.0070
  1569714665   0.0059  -0.0063  -0.0032  +0.0122  +0.0091
  2006902500   0.0079  -0.0022  -0.0050  +0.0101  +0.0129
   342858866   0.0168   0.0050   0.0043  +0.0119  +0.0126
  1591287646  -0.0057  -0.0191  -0.0162  +0.0134  +0.0106
  2763601433   0.0295   0.0186   0.0150  +0.0109  +0.0145
  1524358342   0.0712   0.0560   0.0592  +0.0151  +0.0120


In [16]:
# ---------------------------------------------------------
# Negative-control FNR gaps (10-year bins are the primary comparison)
# ---------------------------------------------------------
def mean_matched_fnrs(ss, sets_by_seed):
    thr = THRESHOLDS[str(ss)]
    female, male = [], []
    for rep in sets_by_seed[ss]:
        sub = Gpid.loc[rep["Patient ID"].to_numpy()]
        ff, mm = fnrs(sub, thr)
        female.append(ff)
        male.append(mm)
    return float(np.nanmean(female)), float(np.nanmean(male))

negative_control_rows = []
print(f"{'seed':>12} {'S_raw':>9} {'S_real':>9} {'S_fake10':>9} {'S_fake5':>9} {'dS_real':>9} {'dS_fake10':>10}")
for ss in ALL_SEEDS:
    fr, mr, s_raw = raw[ss]
    f_real, m_real = matched_fnrs(ss)
    s_real = f_real - m_real
    f10, m10 = mean_matched_fnrs(ss, fake_matched_sets_10yr)
    f5, m5 = mean_matched_fnrs(ss, fake_matched_sets_5yr)
    s10, s5 = f10 - m10, f5 - m5
    negative_control_rows.append({
        "split_seed": ss,
        "FNRf_raw": fr, "FNRm_raw": mr, "S_raw": s_raw,
        "FNRf_real_match": f_real, "FNRm_real_match": m_real, "S_real_match": s_real,
        "FNRf_fake10": f10, "FNRm_fake10": m10, "S_fake10": s10,
        "FNRf_fake5": f5, "FNRm_fake5": m5, "S_fake5": s5,
        "dS_real": s_raw - s_real,
        "dS_fake10": s_raw - s10,
        "dS_fake5": s_raw - s5,
    })
    print(f"{ss:>12} {s_raw:>9.4f} {s_real:>9.4f} {s10:>9.4f} {s5:>9.4f} "
          f"{s_raw-s_real:>+9.4f} {s_raw-s10:>+10.4f}")

negative_control_points = pd.DataFrame(negative_control_rows)
print("\nMean across splits:")
print(negative_control_points[["S_raw", "S_real_match", "S_fake10", "S_fake5",
                               "dS_real", "dS_fake10", "dS_fake5"]].mean().round(5))

        seed     S_raw    S_real  S_fake10   S_fake5   dS_real  dS_fake10
  3658676649    0.0183    0.0086    0.0178    0.0167   +0.0097    +0.0005
   768519171    0.0055   -0.0049    0.0046    0.0050   +0.0104    +0.0009
   113462462    0.0612    0.0519    0.0597    0.0584   +0.0094    +0.0016
  2748406118    0.0602    0.0532    0.0610    0.0628   +0.0069    -0.0008
  1569714665    0.0059   -0.0063    0.0058    0.0068   +0.0122    +0.0002
  2006902500    0.0079   -0.0022    0.0083    0.0080   +0.0101    -0.0004
   342858866    0.0168    0.0050    0.0183    0.0194   +0.0119    -0.0015
  1591287646   -0.0057   -0.0191   -0.0076   -0.0061   +0.0134    +0.0020
  2763601433    0.0295    0.0186    0.0290    0.0319   +0.0109    +0.0006
  1524358342    0.0712    0.0560    0.0721    0.0740   +0.0151    -0.0010

Mean across splits:
S_raw           0.02709
S_real_match    0.01608
S_fake10        0.02690
S_fake5         0.02770
dS_real         0.01101
dS_fake10       0.00020
dS_fake5       -0.000

In [17]:
B, INNER, CL = 1000, 15, 0.95
lo_q, hi_q = 100 * (1 - CL) / 2, 100 * (1 + CL) / 2
rng = np.random.default_rng(20260823)
def ci(v):
    v = np.asarray([x for x in v if np.isfinite(x)])
    return (float(np.percentile(v, lo_q)), float(np.percentile(v, hi_q)))

Yf = [f"y_{f}" for f in NIH_FINDING_NAMES]
Sf = [f"s_{f}" for f in NIH_FINDING_NAMES]

cis = {}; t0 = time.time()
for ss in ALL_SEEDS:
    thr  = np.array([THRESHOLDS[str(ss)][f] for f in NIH_FINDING_NAMES])
    Gt   = test_frame(ss).reset_index(drop=True)
    Gt["weight"] = Gt["Patient ID"].map(ipw_sets[ss].set_index("Patient ID")["weight"]).to_numpy()

    y    = Gt[Yf].to_numpy().astype(bool)
    fnp  = y & (Gt[Sf].to_numpy() < thr)
    sexF = (Gt["sex"].to_numpy() == "F")
    binc = Gt["bin10"].to_numpy()
    wipw = Gt["weight"].to_numpy()
    ubin = np.unique(binc)
    n    = len(Gt)

    def pf(rr, ww):
        if ww is None: P, FN = y[rr].sum(0), fnp[rr].sum(0)
        else:          P, FN = (ww[:, None]*y[rr]).sum(0), (ww[:, None]*fnp[rr]).sum(0)
        tp = P.sum(); return FN.sum()/tp if tp > 0 else np.nan

    def gap(rows, w=None):
        f = sexF[rows]
        if w is None: return pf(rows[f], None) - pf(rows[~f], None)
        return pf(rows[f], w[f]) - pf(rows[~f], w[~f])

    def matched_gap(idx, r):
        bres, fres = binc[idx], sexF[idx]; keep = []
        for b in ubin:
            fb = np.where((bres==b) & fres)[0]; mb = np.where((bres==b) & ~fres)[0]
            k = min(len(fb), len(mb))
            if k == 0: continue
            keep.append(fb if len(fb)==k else r.choice(fb, k, replace=False))
            keep.append(mb if len(mb)==k else r.choice(mb, k, replace=False))
        return gap(idx[np.concatenate(keep)]) if keep else np.nan

    braw=[]; bmat=[]; bipw=[]; bdm=[]; bdi=[]
    for _ in range(B):
        idx = rng.integers(0, n, n)
        sr = gap(idx)
        si = gap(idx, wipw[idx])
        gm = np.mean([matched_gap(idx, rng) for _ in range(INNER)])
        braw.append(sr); bipw.append(si); bmat.append(gm); bdm.append(sr-gm); bdi.append(sr-si)
    cis[ss] = dict(S_raw=ci(braw), S_match=ci(bmat), S_ipw=ci(bipw), dS_match=ci(bdm), dS_ipw=ci(bdi))
    print(f"seed {ss}: done  ({time.time()-t0:.0f}s)")
print(f"total {time.time()-t0:.0f}s")

seed 3658676649: done  (19s)
seed 768519171: done  (39s)
seed 113462462: done  (58s)
seed 2748406118: done  (78s)
seed 1569714665: done  (98s)
seed 2006902500: done  (117s)
seed 342858866: done  (137s)
seed 1591287646: done  (156s)
seed 2763601433: done  (177s)
seed 1524358342: done  (196s)
total 196s


In [27]:
# =========================================================
# Negative-control paired bootstrap
# Tests whether fake-age matching changes the sex gap
# =========================================================

import os
import numpy as np
import pandas as pd

FAKE_BOOTSTRAP_B = 1000
FAKE_INNER_MATCHES = 15  # Matches the existing notebook bootstrap
FAKE_BOOTSTRAP_SEED = 20260826
FAKE_CL = 0.95

fake_rng = np.random.default_rng(FAKE_BOOTSTRAP_SEED)

fake_lo_q = 100 * (1 - FAKE_CL) / 2
fake_hi_q = 100 * (1 + FAKE_CL) / 2

Y_COLS = [f"y_{f}" for f in NIH_FINDING_NAMES]
S_COLS = [f"s_{f}" for f in NIH_FINDING_NAMES]

fake_bootstrap_results = {}
fake_bootstrap_draws = {}

t0 = time.time()

for ss in ALL_SEEDS:

    # Same held-out patients and frozen thresholds as the main analysis
    thr = np.array([
        THRESHOLDS[str(ss)][f]
        for f in NIH_FINDING_NAMES
    ])

    test = test_frame(ss).reset_index(drop=True)

    y = test[Y_COLS].to_numpy().astype(bool)
    scores = test[S_COLS].to_numpy()
    false_negative = y & (scores < thr)

    is_female = test["sex"].to_numpy() == "F"
    fake_bins = test["fake_bin10"].to_numpy()

    unique_fake_bins = np.unique(fake_bins)
    n_patients = len(test)

    def pooled_fnr_rows(rows):
        """Pooled FNR across all 14 findings."""
        positives = y[rows].sum()
        false_negatives = false_negative[rows].sum()

        return (
            false_negatives / positives
            if positives > 0
            else np.nan
        )

    def sex_gap(rows):
        """Female FNR minus male FNR."""
        female_rows = rows[is_female[rows]]
        male_rows = rows[~is_female[rows]]

        female_fnr = pooled_fnr_rows(female_rows)
        male_fnr = pooled_fnr_rows(male_rows)

        return female_fnr - male_fnr

    def fake_age_matched_gap(bootstrap_rows, rng):
        """
        Apply the same exact-bin matching procedure using fake 10-year
        age bins. Keep the smaller sex group and randomly sample the
        larger group without replacement.
        """
        resampled_bins = fake_bins[bootstrap_rows]
        resampled_female = is_female[bootstrap_rows]

        keep_positions = []

        for age_bin in unique_fake_bins:
            female_positions = np.where(
                (resampled_bins == age_bin) & resampled_female
            )[0]

            male_positions = np.where(
                (resampled_bins == age_bin) & ~resampled_female
            )[0]

            n_match = min(
                len(female_positions),
                len(male_positions)
            )

            if n_match == 0:
                continue

            if len(female_positions) == n_match:
                selected_female = female_positions
            else:
                selected_female = rng.choice(
                    female_positions,
                    size=n_match,
                    replace=False
                )

            if len(male_positions) == n_match:
                selected_male = male_positions
            else:
                selected_male = rng.choice(
                    male_positions,
                    size=n_match,
                    replace=False
                )

            keep_positions.extend([
                selected_female,
                selected_male
            ])

        if not keep_positions:
            return np.nan

        matched_positions = np.concatenate(keep_positions)
        matched_rows = bootstrap_rows[matched_positions]

        return sex_gap(matched_rows)

    raw_draws = []
    fake_draws = []
    change_draws = []

    for _ in range(FAKE_BOOTSTRAP_B):

        # Patient-level bootstrap with replacement
        bootstrap_rows = fake_rng.integers(
            0,
            n_patients,
            size=n_patients
        )

        raw_gap = sex_gap(bootstrap_rows)

        # Average repeated fake-age matching realizations, following
        # the same convention as the notebook's existing bootstrap.
        fake_gap = np.nanmean([
            fake_age_matched_gap(bootstrap_rows, fake_rng)
            for _ in range(FAKE_INNER_MATCHES)
        ])

        change = raw_gap - fake_gap

        raw_draws.append(raw_gap)
        fake_draws.append(fake_gap)
        change_draws.append(change)

    raw_draws = np.asarray(raw_draws)
    fake_draws = np.asarray(fake_draws)
    change_draws = np.asarray(change_draws)

    finite_change = change_draws[np.isfinite(change_draws)]

    change_ci = np.percentile(
        finite_change,
        [fake_lo_q, fake_hi_q]
    )

    point_row = negative_control_points.loc[
        negative_control_points["split_seed"] == ss
    ].iloc[0]

    point_change = float(point_row["dS_fake10"])
    ci_contains_zero = bool(
        change_ci[0] <= 0 <= change_ci[1]
    )

    fake_bootstrap_results[ss] = {
        "split_seed": ss,
        "S_raw": float(point_row["S_raw"]),
        "S_fake10": float(point_row["S_fake10"]),
        "dS_fake10": point_change,
        "ci_lower": float(change_ci[0]),
        "ci_upper": float(change_ci[1]),
        "ci_contains_zero": ci_contains_zero,
    }

    fake_bootstrap_draws[ss] = change_draws

    print(
        f"seed {ss}: "
        f"dS_fake10={point_change:+.5f}, "
        f"95% CI=[{change_ci[0]:+.5f}, {change_ci[1]:+.5f}], "
        f"contains zero={ci_contains_zero}"
    )

print(f"\nBootstrap finished in {time.time() - t0:.1f} seconds")

fake_ci_df = pd.DataFrame(
    fake_bootstrap_results.values()
)

print("\nPer-split negative-control results:")
print(fake_ci_df.to_string(index=False))


# ---------------------------------------------------------
# Mean-across-splits negative-control estimate and CI
# ---------------------------------------------------------

# Combine corresponding bootstrap draws across the 10 splits.
draw_matrix = np.vstack([
    fake_bootstrap_draws[ss]
    for ss in ALL_SEEDS
])

mean_change_draws = np.nanmean(
    draw_matrix,
    axis=0
)

mean_point_change = float(
    negative_control_points["dS_fake10"].mean()
)

mean_ci = np.percentile(
    mean_change_draws[np.isfinite(mean_change_draws)],
    [fake_lo_q, fake_hi_q]
)

mean_ci_contains_zero = bool(
    mean_ci[0] <= 0 <= mean_ci[1]
)

print("\n" + "=" * 60)
print("NEGATIVE-CONTROL CONCLUSION")
print("=" * 60)
print(f"Mean dS_fake10: {mean_point_change:+.5f}")
print(
    f"95% CI: [{mean_ci[0]:+.5f}, {mean_ci[1]:+.5f}]"
)
print(
    "CI contains zero:",
    mean_ci_contains_zero
)

if mean_ci_contains_zero:
    print(
        "\nPASS: Fake-age matching did not significantly change "
        "the sex gap. There is no evidence that the matching "
        "procedure itself introduced an artifact."
    )
else:
    print(
        "\nDEBUG REQUIRED: The fake-age change CI excludes zero. "
        "The matching procedure may be introducing an artifact "
        "and should be investigated before Week 3."
    )


# ---------------------------------------------------------
# Save the results
# ---------------------------------------------------------

NEG_OUT = f"{RESULTS}/group_b_densenet"
os.makedirs(NEG_OUT, exist_ok=True)

fake_ci_df.to_csv(
    f"{NEG_OUT}/group_b_negative_control_bootstrap_ci.csv",
    index=False
)

pd.DataFrame({
    "metric": ["mean_dS_fake10"],
    "estimate": [mean_point_change],
    "ci_lower": [float(mean_ci[0])],
    "ci_upper": [float(mean_ci[1])],
    "ci_contains_zero": [mean_ci_contains_zero],
    "bootstrap_B": [FAKE_BOOTSTRAP_B],
    "inner_matches": [FAKE_INNER_MATCHES],
    "bootstrap_seed": [FAKE_BOOTSTRAP_SEED],
}).to_csv(
    f"{NEG_OUT}/group_b_negative_control_conclusion.csv",
    index=False
)

print(
    "\nSaved:",
    f"{NEG_OUT}/group_b_negative_control_bootstrap_ci.csv"
)

print(
    "Saved:",
    f"{NEG_OUT}/group_b_negative_control_conclusion.csv"
)

seed 3658676649: dS_fake10=+0.00052, 95% CI=[-0.00483, +0.00505], contains zero=True
seed 768519171: dS_fake10=+0.00090, 95% CI=[-0.00412, +0.00565], contains zero=True
seed 113462462: dS_fake10=+0.00157, 95% CI=[-0.00380, +0.00505], contains zero=True
seed 2748406118: dS_fake10=-0.00085, 95% CI=[-0.00517, +0.00320], contains zero=True
seed 1569714665: dS_fake10=+0.00018, 95% CI=[-0.00543, +0.00362], contains zero=True
seed 2006902500: dS_fake10=-0.00040, 95% CI=[-0.00467, +0.00441], contains zero=True
seed 342858866: dS_fake10=-0.00149, 95% CI=[-0.00479, +0.00390], contains zero=True
seed 1591287646: dS_fake10=+0.00197, 95% CI=[-0.00285, +0.00692], contains zero=True
seed 2763601433: dS_fake10=+0.00056, 95% CI=[-0.00530, +0.00377], contains zero=True
seed 1524358342: dS_fake10=-0.00098, 95% CI=[-0.00585, +0.00316], contains zero=True

Bootstrap finished in 126.8 seconds

Per-split negative-control results:
 split_seed     S_raw  S_fake10  dS_fake10  ci_lower  ci_upper  ci_contains_zer

In [18]:
img = pd.DataFrame({
    "Image Index": meta[COL_IMAGE].astype(str).str.strip(),
    "Patient ID":  meta[COL_PATIENT].astype(str).str.strip(),
    "sex":         meta[COL_SEX].astype(str).str.strip().str.upper().str[0],
}).merge(S, on="Image Index", how="inner")
fl_all = meta[lab_col].fillna("").astype(str).str.split("|")
for f in NIH_FINDING_NAMES:
    img[f"y_{f}"] = fl_all.apply(lambda labs: int(f in labs))
img = img[img["sex"].isin(["F","M"])].reset_index(drop=True)

def pf_img(d, thr, sx):
    d = d[d["sex"]==sx]; FN=P=0
    for f in NIH_FINDING_NAMES:
        y=d[f"y_{f}"].to_numpy(); s=d[f"s_{f}"].to_numpy(); pos=(y==1)
        P+=pos.sum(); FN+=(pos&(s<thr[f])).sum()
    return FN/P if P else np.nan

print(f"{'seed':>12} {'n_1st':>7} {'n_all':>7} {'S_1st':>9} {'S_all':>9} {'GroupA':>9}")
for ss in ALL_SEEDS:
    ids = test_ids(ss); da = img[img["Patient ID"].isin(ids)]; thr = THRESHOLDS[str(ss)]
    sa = pf_img(da, thr, "F") - pf_img(da, thr, "M")
    print(f"{ss:>12} {sum(G['Patient ID'].isin(ids)):>7} {len(da):>7} "
          f"{raw[ss][2]:>9.4f} {sa:>9.4f} {float(ga_gap.get(ss, np.nan)):>9.4f}")

        seed   n_1st   n_all     S_1st     S_all    GroupA
  3658676649    6158   22489    0.0183    0.0393    0.0393
   768519171    6160   22358    0.0055    0.0196    0.0196
   113462462    6159   22400    0.0612    0.0172    0.0172
  2748406118    6160   22134    0.0602    0.0386    0.0386
  1569714665    6160   22333    0.0059    0.0257    0.0257
  2006902500    6159   22060    0.0079   -0.0011   -0.0011
   342858866    6159   22035    0.0168    0.0255    0.0255
  1591287646    6158   22369   -0.0057    0.0338    0.0338
  2763601433    6160   22278    0.0295   -0.0092   -0.0092
  1524358342    6161   22181    0.0712    0.0377    0.0377


In [21]:
import os
DATASET, BACKBONE = "NIH", "densenet121-res224-all"
OUT = f"{RESULTS}/group_b_densenet"; os.makedirs(OUT, exist_ok=True)

def emit(cond, r, c):
    m = c.replace("_", "")
    g = r[f"S_{c}"]; gc = cis[r.seed][f"S_{c}"]; ds = r[f"dS_{c}"]; dc = cis[r.seed][f"dS_{c}"]
    base_ = dict(dataset=DATASET, backbone=BACKBONE, condition=cond, split_seed=r.seed, finding="NIH_14_pooled")
    return [
      {**base_, "subgroup":"sex:female", "metric":"fnr", "value":r[f"FNRf_{c}"], "ci_lower":np.nan, "ci_upper":np.nan},
      {**base_, "subgroup":"sex:male",   "metric":"fnr", "value":r[f"FNRm_{c}"], "ci_lower":np.nan, "ci_upper":np.nan},
      {**base_, "subgroup":"sex_gap:female-minus-male", "metric":"fnr", "value":g, "ci_lower":gc[0], "ci_upper":gc[1]},
      {**base_, "subgroup":"sex_gap:female-minus-male", "metric":"fnr_gap_reduction", "value":ds, "ci_lower":dc[0], "ci_upper":dc[1]},
    ]

tidy = []
for _, r in pts.iterrows():
    tidy += emit("condition_2_agestd_match", r, "match")
    tidy += emit("condition_3_agestd_ipw",   r, "ipw")
tidy = pd.DataFrame(tidy)
tidy.to_csv(f"{OUT}/group_b_densenet_agestd_with_ci.csv", index=False)
pts.to_csv(f"{OUT}/group_b_densenet_agestd_points.csv", index=False)

summary = pts[["S_raw","S_match","S_ipw","dS_match","dS_ipw"]].mean().round(5)
summary.to_csv(f"{OUT}/group_b_summary_across_splits.csv")
print("HEADLINE (mean across 10 splits):")
print(summary.to_string())

pd.Series(MATCH_SEEDS).to_csv(
    os.path.join(OUT, "matching_seeds_for_commit.txt"),
    index=False,
    header=False
)
json.dump({"dataset":DATASET,"backbone":BACKBONE,"split_seeds":list(map(int,ALL_SEEDS)),
           "primary_bin":"bin10","match_seed_base":int(MATCH_SEED_BASE),"n_match_seeds":int(N_MATCH_SEEDS),
           "bootstrap_B":B,"inner_matches":INNER,"thresholds_source":"group_a thresholds_by_seed.json",
           "eval_basis":"first_index_scan_per_patient","weights":"uncapped, held fixed in bootstrap"},
          open(f"{OUT}/run_manifest.json","w"), indent=2)
print("\nwrote ->", OUT)

HEADLINE (mean across 10 splits):
S_raw       0.02709
S_match     0.01608
S_ipw       0.01626
dS_match    0.01101
dS_ipw      0.01083

wrote -> /content/drive/MyDrive/group_b_densenet


In [22]:
# ---------------------------------------------------------
# Export negative-control results and frozen run metadata
# ---------------------------------------------------------
NEG_OUT = f"{RESULTS}/group_b_densenet"
os.makedirs(NEG_OUT, exist_ok=True)

negative_control_points.to_csv(
    f"{NEG_OUT}/group_b_negative_control_points.csv", index=False
)

fake_match_diagnostics = []
for width, sets_by_seed in [(10, fake_matched_sets_10yr), (5, fake_matched_sets_5yr)]:
    for ss, reps in sets_by_seed.items():
        for match_rep, rep in enumerate(reps):
            fake_match_diagnostics.append({
                "bin_width": width,
                "split_seed": ss,
                "match_rep": match_rep,
                "match_seed": MATCH_SEEDS[match_rep],
                "n_matched": len(rep),
                "n_female": int((rep.sex == "F").sum()),
                "n_male": int((rep.sex == "M").sum()),
            })
fake_match_diagnostics = pd.DataFrame(fake_match_diagnostics)
assert (fake_match_diagnostics["n_female"] == fake_match_diagnostics["n_male"]).all()
fake_match_diagnostics.to_csv(
    f"{NEG_OUT}/group_b_negative_control_matching_diagnostics.csv", index=False
)

json.dump({
    "analysis": "negative-control age-permutation resampler",
    "fake_age_seed": int(FAKE_AGE_SEED),
    "split_seeds": list(map(int, ALL_SEEDS)),
    "match_seed_base": int(MATCH_SEED_BASE),
    "n_match_seeds": int(N_MATCH_SEEDS),
    "primary_fake_bin": "fake_bin10",
    "sensitivity_fake_bin": "fake_bin5",
    "permutation_scope": "one global patient-level permutation before frozen test subsets",
    "evaluation_basis": "first index scan per patient",
    "gap_definition": "FNR_female - FNR_male",
}, open(f"{NEG_OUT}/negative_control_manifest.json", "w"), indent=2)

print("wrote ->", f"{NEG_OUT}/group_b_negative_control_points.csv")
print("wrote ->", f"{NEG_OUT}/group_b_negative_control_matching_diagnostics.csv")
print("wrote ->", f"{NEG_OUT}/negative_control_manifest.json")

wrote -> /content/drive/MyDrive/group_b_densenet/group_b_negative_control_points.csv
wrote -> /content/drive/MyDrive/group_b_densenet/group_b_negative_control_matching_diagnostics.csv
wrote -> /content/drive/MyDrive/group_b_densenet/negative_control_manifest.json


In [23]:

print("matched_sets keys:", list(matched_sets)[:2], "...")
r0 = matched_sets[ALL_SEEDS[0]][0]
print("one replicate cols:", r0.columns.tolist())
print("n replicates per seed:", len(matched_sets[ALL_SEEDS[0]]))


w0 = ipw_sets[ALL_SEEDS[0]]
print("ipw cols:", w0.columns.tolist(), "| rows:", len(w0))

matched_sets keys: [3658676649, 768519171] ...
one replicate cols: ['Patient ID', 'Image Index', 'sex', 'age', 'followup', 'bin5', 'bin10', 'fake_age', 'fake_bin5', 'fake_bin10']
n replicates per seed: 100
ipw cols: ['Patient ID', 'Image Index', 'sex', 'age', 'followup', 'bin5', 'bin10', 'fake_age', 'fake_bin5', 'fake_bin10', 'weight'] | rows: 6158


In [24]:

import numpy as np, pandas as pd, os
OUT = f"{RESULTS}/group_b_densenet"; os.makedirs(OUT, exist_ok=True)
Yf = [f"y_{f}" for f in NIH_FINDING_NAMES]; Sf = [f"s_{f}" for f in NIH_FINDING_NAMES]


Gr = G.reset_index(drop=True)
pid_i = {p: i for i, p in enumerate(Gr["Patient ID"])}
Ymat = Gr[Yf].to_numpy().astype(bool); Smat = Gr[Sf].to_numpy(); sexA = Gr["sex"].to_numpy()

pt_rows, cnt_rows = [], []
for ss in ALL_SEEDS:
    thr = np.array([THRESHOLDS[str(ss)][f] for f in NIH_FINDING_NAMES])
    ipw = ipw_sets[ss][["Patient ID", "weight"]]
    test_pids = ipw["Patient ID"].to_numpy()

    from collections import Counter
    c = Counter()
    for rep in matched_sets[ss]:
        c.update(rep["Patient ID"].tolist())
    nrep = len(matched_sets[ss])

    sub = (G[G["Patient ID"].isin(set(test_pids))]
             .merge(ipw, on="Patient ID", how="left"))
    sub["matched_frac"] = sub["Patient ID"].map(lambda p: c.get(p, 0) / nrep)
    sub.insert(0, "split_seed", ss)
    pt_rows.append(sub[["split_seed", "Patient ID", "Image Index", "sex", "age", "bin10",
                        *Yf, *Sf, "weight", "matched_frac"]].rename(columns={"weight": "ipw_weight"}))

    def counts(mask_rows, thr, w=None, seed=ss, cond=""):
        out = []
        for sx in ["F", "M"]:
            m = mask_rows & (sexA == sx)
            ww = (np.ones(m.sum()) if w is None else w[m[mask_rows]]) if w is not None else np.ones(m.sum())
            Y = Ymat[m]; S = Smat[m]; ww = np.ones(Y.shape[0]) if w is None else ww
            Pp = FNp = 0.0
            for j, f in enumerate(NIH_FINDING_NAMES):
                pos = Y[:, j]; P = (ww * pos).sum(); FN = (ww * (pos & (S[:, j] < thr[j]))).sum()
                out.append((seed, f, sx, cond, P, FN, FN / P if P else np.nan, thr[j], w is not None))
                Pp += P; FNp += FN
            out.append((seed, "NIH_14_pooled", sx, cond, Pp, FNp, FNp / Pp if Pp else np.nan, np.nan, w is not None))
        return out

    test_mask = np.array([p in set(test_pids) for p in Gr["Patient ID"]])
    cnt_rows += counts(test_mask, thr, w=None, cond="raw")


    wmap = dict(zip(ipw["Patient ID"], ipw["weight"]))
    wvec = np.array([wmap.get(p, np.nan) for p in Gr["Patient ID"][test_mask]])
    cnt_rows += counts(test_mask, thr, w=wvec, cond="ipw")


    acc = {}
    for rep in matched_sets[ss]:
        idx = np.array([pid_i[p] for p in rep["Patient ID"]])
        for sx in ["F", "M"]:
            mm = idx[sexA[idx] == sx]; Y = Ymat[mm]; S = Smat[mm]
            for j, f in enumerate(NIH_FINDING_NAMES):
                pos = Y[:, j]; P = pos.sum(); FN = (pos & (S[:, j] < thr[j])).sum()
                a = acc.setdefault((f, sx), [0.0, 0.0]); a[0] += P; a[1] += FN
    for (f, sx), (sumP, sumFN) in acc.items():
        P, FN = sumP / nrep, sumFN / nrep
        cnt_rows.append((ss, f, sx, "matched", P, FN, FN / P if P else np.nan, np.nan, False))

patient_level = pd.concat(pt_rows, ignore_index=True)
counts_df = pd.DataFrame(cnt_rows, columns=["split_seed","finding","sex","condition",
                        "positives","false_negatives","fnr","threshold","weighted"])
patient_level.to_parquet(f"{OUT}/group_b_patient_level.parquet", index=False)
counts_df.to_parquet(f"{OUT}/group_b_counts_by_finding.parquet", index=False)
print("patient_level:", patient_level.shape, "->", f"{OUT}/group_b_patient_level.parquet")
print("counts:", counts_df.shape, "->", f"{OUT}/group_b_counts_by_finding.parquet")
print(counts_df[counts_df.finding=="NIH_14_pooled"].head(6).to_string(index=False))

patient_level: (61594, 36) -> /content/drive/MyDrive/group_b_densenet/group_b_patient_level.parquet
counts: (880, 9) -> /content/drive/MyDrive/group_b_densenet/group_b_counts_by_finding.parquet
 split_seed       finding sex condition   positives  false_negatives      fnr  threshold  weighted
 3658676649 NIH_14_pooled   F       raw 1146.000000       576.000000 0.502618        NaN     False
 3658676649 NIH_14_pooled   M       raw 1462.000000       708.000000 0.484268        NaN     False
 3658676649 NIH_14_pooled   F       ipw 1169.425922       583.411193 0.498887        NaN      True
 3658676649 NIH_14_pooled   M       ipw 1437.614599       703.230905 0.489165        NaN      True
  768519171 NIH_14_pooled   F       raw 1132.000000       571.000000 0.504417        NaN     False
  768519171 NIH_14_pooled   M       raw 1419.000000       708.000000 0.498943        NaN     False


In [25]:
import numpy as np
for k in ["S_raw","S_match","S_ipw","dS_match","dS_ipw"]:
    lo = np.mean([cis[ss][k][0] for ss in ALL_SEEDS])
    hi = np.mean([cis[ss][k][1] for ss in ALL_SEEDS])
    print(f"{k:9s}  [{lo:+.4f}, {hi:+.4f}]")

S_raw      [-0.0155, +0.0699]
S_match    [-0.0270, +0.0591]
S_ipw      [-0.0263, +0.0596]
dS_match   [+0.0038, +0.0191]
dS_ipw     [+0.0066, +0.0149]


In [26]:
c = counts_df[(counts_df.finding=="NIH_14_pooled") & (counts_df.condition=="raw")]
for ss in ALL_SEEDS[:3]:
    ex_f = c[(c.split_seed==ss)&(c.sex=="F")]["fnr"].values[0]
    ex_m = c[(c.split_seed==ss)&(c.sex=="M")]["fnr"].values[0]
    print(f"{ss}: export F={ex_f:.4f} M={ex_m:.4f} | Cell3 F={raw[ss][0]:.4f} M={raw[ss][1]:.4f}")

3658676649: export F=0.5026 M=0.4843 | Cell3 F=0.5026 M=0.4843
768519171: export F=0.5044 M=0.4989 | Cell3 F=0.5044 M=0.4989
113462462: export F=0.5127 M=0.4515 | Cell3 F=0.5127 M=0.4515
